# 🧬 GenomeScan AI: Multi-Class Disease Prediction Training

This notebook trains DNABERT-2 to classify pathogenic variants into one of 15 disease categories.

In [ ]:
!pip install -q --force-reinstall transformers==4.44.0
!pip install -q datasets accelerate pyfaidx scikit-learn einops
!pip install -q triton 2>/dev/null || echo 'triton not available, continuing without flash attention'

In [ ]:
import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import Dataset
from transformers import AutoModel, AutoConfig, AutoTokenizer, TrainingArguments, Trainer
from transformers.modeling_outputs import SequenceClassifierOutput
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from pyfaidx import Fasta
from google.colab import files

SEED = 42
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
print('Please upload clinvar_disease_snps.csv and disease_label_map.csv')
uploaded = files.upload()

df = pd.read_csv('clinvar_disease_snps.csv')
label_map_df = pd.read_csv('disease_label_map.csv')
print(f'✓ Loaded dataset: {len(df):,} samples over {len(label_map_df)} categories.')

In [ ]:
print('Downloading GRCh38 Genome...')
if not os.path.exists('hg38.fa.gz'):
    !wget -q https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz
if not os.path.exists('hg38.fa'):
    !gunzip hg38.fa.gz

print('Indexing genome...')
genome = Fasta('hg38.fa', rebuild=False)
print('✓ Genome ready.')

In [ ]:
CONTEXT = 64
VALID_BASES = set('ACGT')
genome_keys = set(genome.keys())

def resolve_chrom(chrom_raw):
    c = str(chrom_raw)
    if c in genome_keys: return c
    if f'chr{c}' in genome_keys: return f'chr{c}'
    if c == 'MT' and 'chrM' in genome_keys: return 'chrM'
    return None

def extract_pair(row):
    chrom_key = resolve_chrom(row['Chromosome'])
    if chrom_key is None: return None, None
    
    pos = int(row['Start']) - 1
    ref_allele = str(row['ReferenceAllele']).upper()
    alt_allele = str(row['AlternateAllele']).upper()
    
    if ref_allele not in VALID_BASES or alt_allele not in VALID_BASES: return None, None
    
    chrom_seq = genome[chrom_key]
    start, end = pos - CONTEXT, pos + CONTEXT + 1
    if start < 0 or end > len(chrom_seq): return None, None
    
    left = str(chrom_seq[start:pos]).upper()
    right = str(chrom_seq[pos + 1:end]).upper()
    
    ref_seq, alt_seq = left + ref_allele + right, left + alt_allele + right
    if not all(b in VALID_BASES for b in ref_seq): return None, None
    return ref_seq, alt_seq

print('Extracting reference + mutant sequence pairs...')
results = df.apply(extract_pair, axis=1, result_type='expand')
df['ref_seq'], df['alt_seq'] = results[0], results[1]
df = df.dropna(subset=['ref_seq', 'alt_seq']).reset_index(drop=True)

train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['disease_label'], random_state=SEED)
print(f'✓ Extraction complete! Train: {len(train_df):,} | Test: {len(test_df):,}')

In [ ]:
MODEL_NAME = "zhihan1996/DNABERT-2-117M"
MAX_LENGTH = 256

print('Loading DNABERT-2 tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

class DNADiseaseDataset(Dataset):
    def __init__(self, ref_seqs, alt_seqs, labels):
        self.encodings = tokenizer(ref_seqs, alt_seqs, max_length=MAX_LENGTH, padding='max_length', truncation=True, return_tensors='pt')
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

train_dataset = DNADiseaseDataset(train_df['ref_seq'].tolist(), train_df['alt_seq'].tolist(), train_df['disease_label'].tolist())
test_dataset = DNADiseaseDataset(test_df['ref_seq'].tolist(), test_df['alt_seq'].tolist(), test_df['disease_label'].tolist())

In [ ]:
import transformers.models.auto.auto_factory as _af
_orig_register = _af._BaseAutoModelClass.register.__func__
@classmethod
def _safe_register(cls, config_class, model_class, exist_ok=False):
    return _orig_register(cls, config_class, model_class, exist_ok=True)
_af._BaseAutoModelClass.register = _safe_register

NUM_LABELS = len(label_map_df)

class DNABertDiseaseClassifier(nn.Module):
    def __init__(self, model_name, num_labels=NUM_LABELS, dropout=0.3):
        super().__init__()
        config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
        if not hasattr(config, 'pad_token_id'): config.pad_token_id = 0
        
        self.encoder = AutoModel.from_pretrained(
            model_name, config=config, trust_remote_code=True, 
            ignore_mismatched_sizes=True, _fast_init=False, low_cpu_mem_usage=False
        )
        h = self.encoder.config.hidden_size
        self.head = nn.Sequential(
            nn.Linear(h, 256), nn.LayerNorm(256), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(256, num_labels)
        )
        self.config = self.encoder.config

    def forward(self, input_ids, attention_mask=None, labels=None, **kw):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_emb = out.last_hidden_state[:, 0]
        logits = self.head(cls_emb)
        loss = nn.CrossEntropyLoss()(logits, labels) if labels is not None else None
        return SequenceClassifierOutput(loss=loss, logits=logits)
        
    def save_pretrained(self, path):
        os.makedirs(path, exist_ok=True)
        self.encoder.save_pretrained(os.path.join(path, 'encoder'))
        torch.save(self.head.state_dict(), os.path.join(path, 'classifier_head.pt'))
        self.config.save_pretrained(path)

model = DNABertDiseaseClassifier(MODEL_NAME).to(device)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro')
    }

for param in model.parameters(): param.requires_grad = True

training_args = TrainingArguments(
    output_dir='./dnabert2_disease_final',
    num_train_epochs=5,  # 5 epochs for multi-class
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=3e-5,
    weight_decay=0.01,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    report_to='none'
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_dataset, eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print('🚀 Starting Disease Model Training...')
trainer.train()

In [ ]:
print('Saving and Zipping model...')
model.save_pretrained('./dnabert2_disease_final')
!zip -r dnabert2_disease_model.zip dnabert2_disease_final/
files.download('dnabert2_disease_model.zip')